In [109]:
import time
import pickle
import numpy as np
import scipy
from scipy.linalg import sqrtm, inv
from sklearn.covariance import GraphicalLassoCV
from tqdm import tqdm
from functions.gram_matrix import gram_matrix, gram_matrix2, gram_matrix3
from functions.lib_fun import gwire_cv
from functions.FOPG import FOPG
from functions.FD_SDR import FD_SDR


def orth_mat(x):
    temp = x.T @ x 
    C = temp**(-1/2)
    return x*C

def direction_error_angle(b, beta_true):
    """Calculate the directional error between estimated and true coefficients."""
    proj_b = b @ inv(b.T @ b) @ b.T
    proj_true = beta_true @ inv(beta_true.T @ beta_true) @ beta_true.T
    return np.linalg.norm(proj_b - proj_true, 'fro')


def generate_X(n, p, rho = 0.5, mode_X = '(a)'):
    """Data generation for X"""

    if p <= 5:
        raise ValueError("Dimension p must be greater than or equal to 6")
    if mode_X not in ['(a)', '(b)']:
        raise ValueError("Mode must be one of '(a)', '(b)'")
    
    if mode_X == '(a)':
        # X = np.random.randn(n, p)
        X = scipy.stats.norm.cdf(np.random.randn(n, p))
    elif mode_X == '(b)':
        X = np.zeros((n, p))
        X[:, 0] = np.random.normal(size=n) 
        for i in range(1, p):
            X[:, i] = rho * X[:, i - 1] + np.sqrt(1 - rho**2) * np.random.normal(size=n)

        X = scipy.stats.norm.cdf(X)
    return X

def gendata_sphere_Y(X, model_Y='(1)'):
    """Data generation for Y"""

    n = X.shape[0]
    p = X.shape[1]
 
    if model_Y not in ['(1)', '(2)', '(3)']:
        raise ValueError("Model must be one of '(1)', '(2)', '(3)'")
    
    beta_1 = np.random.normal(0, 1, size=p) / np.sqrt(3)
    beta_1[3:] = 0
    beta_2 = np.random.normal(0, 1, size=p) / np.sqrt(3)
    beta_2[:-3] = 0
    beta_3 = np.concatenate([[0, 1], np.zeros(p - 2)])

    if model_Y == '(1)':
        d_0 = 1
        beta_true = beta_1.reshape((p,1)) # p by 1 vector
        X_beta = X @ beta_true

        delta = np.random.normal(0, 0.2, size=n).reshape((n,1)) # p by 1 vector
        m_X = np.column_stack((np.cos(np.pi * X_beta), np.sin(np.pi * X_beta)))  # p by 2 vector
        eps_X = np.column_stack((-delta * np.sin(np.pi * X_beta), delta * np.cos(np.pi * X_beta))) # p by 2 vector
        eps_X_norms = np.linalg.norm(eps_X, axis=1, keepdims=True) # equal to np.abs(delta)

        Y = np.cos(eps_X_norms) * m_X + np.sin(eps_X_norms) * (eps_X / (eps_X_norms + 1e-8))
    
    elif model_Y == '(2)':
        d_0 = 2
        beta_true = np.vstack([beta_1,beta_3]).T
    
        def genone(x, beta1):
            delta = np.random.normal(0, 0.2, size=2)
            x_beta1 = (x @ beta1.T).item() 
        
            m_x = np.array([
                np.sqrt(np.maximum(0, 1 - (x[1])**2)) * np.cos(np.pi * (x_beta1)),
                np.sqrt(np.maximum(0, 1 - (x[1])**2)) * np.sin(np.pi * (x_beta1)),
                (x[1])
                ])
            temp = np.array([[-np.sqrt(1 - (x[1])**2) * np.sin(np.pi * (x_beta1)), 
                          np.sqrt(1 - (x[1])**2) * np.cos(np.pi * (x_beta1)), 0]])
            temp = temp.flatten()
            v1 = orth_mat(temp)
            v2 = orth_mat(np.cross(v1, m_x))
            eps = delta[0] * v1 + delta[1] * v2
            y = np.cos(np.sqrt(np.sum(eps**2))) * m_x + np.sin(np.sqrt(np.sum(eps**2))) / np.sqrt(np.sum(eps**2)) * eps
            return np.concatenate((x, y))

        data_full = np.array([genone(X[i], beta_1) for i in range(n)])
        Y = data_full[:, p:p + 3]
        
    elif model_Y == '(3)':
        d_0 = 2
        beta_true = np.vstack([beta_1,beta_2]).T

        delta_1 = np.random.normal(0, 0.2, size=(n,1))  
        delta_2 = np.random.normal(0, 0.2, size=(n,1))

        X_beta_delta_1 = X @ beta_1.reshape((p,1)) + delta_1
        X_beta_delta_2 = X @ beta_2.reshape((p,1)) + delta_2  

        Y = np.column_stack((
            np.sin(X_beta_delta_1) * np.sin(X_beta_delta_2),
            np.sin(X_beta_delta_1) * np.cos(X_beta_delta_2),
            np.cos(X_beta_delta_1)
            ))  

    return {'X': X, 'y': Y, 'beta_true': beta_true, 'd_0': d_0}

In [113]:
def run_simulation(config):
    num_repeats = config['num_repeats']
    n = config['n']
    p = config['p']
    mode_X = config['mode_X']
    mode_y = config['mode_y']
    IF_GWIRE = config['IF_GWIRE']
    verbose = config['verbose']
    metric = config['metric']
    
    # Initialize result storage
    results = {
        'gwire_times': [],
        'fopg_times': [],
        'fd_sdr_times': [],
        'gwire_errors': [],
        'fd_sdr_errors': [],
        'fopg_errors': []
    }

    """Run simulation"""
    if verbose:
        print("\n" + "="*60)
        print(f"Starting Simulation".center(60))
        print("="*60)
        print(f"Configuration:")
        print(f"- Repeats: {num_repeats}")
        print(f"- Dimensions: n={n}, p={p}")
        print(f"- X mode: {mode_X}, Y mode: {mode_y}")
        print(f"- Metric: {metric}")
        print("="*60 + "\n")

    # Use tqdm for progress bar if verbose is False
    iterator = range(num_repeats)
    if not verbose:
        print(f"Parameters: n = {n}, p = {p}, mode_X = {mode_X}, mode_y = {mode_y}")
        iterator = tqdm(iterator, desc="Running simulation", unit="iter")


    for i in iterator:
        iter_start = time.time()
        if verbose:
            print(f"\nIteration {i+1}/{num_repeats} ".ljust(30, '-'))

        X = generate_X(n, p, mode_X)
        DATA_XY = gendata_sphere_Y(X, mode_y)
        y = DATA_XY['y']
        beta_true = DATA_XY['beta_true']
        d_0 = DATA_XY['d_0']

        iter_start = time.time()

        # GWIRE method
        if IF_GWIRE:
            start_time = time.time()

            glassomodel = GraphicalLassoCV()
            glassomodel.fit(X)
            omega = glassomodel.precision_
            np.where(np.sum(omega!=0, axis = 1) > 1)[0]
            Nb = []
            for i in range(p):
                Ni = (np.nonzero(omega[i,:])[0]).tolist() #np.nonzero(ome[i,:])[0]
                Nb.append(Ni)
            beta_gwire, _ = gwire_cv(X, y, Nb, metric, d_0, fold=5)
            gwire_time = time.time() - start_time
            results['gwire_times'].append(gwire_time)
            gwire_error = direction_error_angle(beta_gwire, beta_true)
            results['gwire_errors'].append(gwire_error)
            if verbose:
                print(f"GWIRE: Time={gwire_time:.3f}s, Error={gwire_error:.4f}")
        
        # FOPG method
        start_time = time.time()
        ygram = gram_matrix3(y,1)
        beta_fopg = FOPG(X, ygram, d_0)
        fopg_time = time.time() - start_time
        results['fopg_times'].append(fopg_time)
        fopg_error = direction_error_angle(beta_fopg, beta_true)
        results['fopg_errors'].append(fopg_error)
        if verbose:
            print(f"FOPG:  Time={fopg_time:.3f}s, Error={fopg_error:.4f}")

        # FD-SDR method
        start_time = time.time()
        ygram2 = gram_matrix3(y,10)
        ygram2 = np.real(sqrtm(ygram2))
        beta_fd_sdr, _, _ = FD_SDR(X.T, ygram2, beta_fopg)
        fd_sdr_time = time.time() - start_time
        results['fd_sdr_times'].append(fd_sdr_time)
        fd_sdr_error = direction_error_angle(beta_fd_sdr, beta_true)
        results['fd_sdr_errors'].append(fd_sdr_error)
        if verbose:
            print(f"FD-SDR: Time={fd_sdr_time:.3f}s, Error={fd_sdr_error:.4f}")

        # Print iteration summary
        iter_time = time.time() - iter_start
        if verbose:
            print("-"*40)
            print(f"Iteration completed in {iter_time:.2f} seconds")
            print("-"*40)

    return results

def print_summary(results, config=None):
    """
    Print a well-formatted summary of the simulation results.
    
    Parameters:
    -----------
    results : dict
        Dictionary containing the simulation results with keys:
        - 'gwire_times', 'fopg_times', 'fd_sdr_times' (lists of times)
        - 'gwire_errors', 'fopg_errors', 'fd_sdr_errors' (lists of errors)
        
    config : dict, optional
        Dictionary containing simulation configuration parameters
    """
    # Header
    print("\n" + "="*80)
    print(" SIMULATION SUMMARY ".center(80, '='))
    print("="*80)
    
    # Print configuration if provided
    if config:
        print("\nCONFIGURATION:")
        for key, value in config.items():
            print(f"- {key}: {value}")
        print("-"*80)
    
    # Calculate statistics
    stats = {}
    methods = []
    
    if 'gwire_times' in results and len(results['gwire_times']) > 0:
        methods.append('GWIRE')
        stats['GWIRE'] = {
            'time_mean': np.mean(results['gwire_times']),
            'time_std': np.std(results['gwire_times']),
            'error_mean': np.mean(results['gwire_errors']),
            'error_std': np.std(results['gwire_errors'])
        }
    
    methods.extend(['FOPG', 'FD-SDR'])
    
    stats['FOPG'] = {
        'time_mean': np.mean(results['fopg_times']),
        'time_std': np.std(results['fopg_times']),
        'error_mean': np.mean(results['fopg_errors']),
        'error_std': np.std(results['fopg_errors'])
    }
    
    stats['FD-SDR'] = {
        'time_mean': np.mean(results['fd_sdr_times']),
        'time_std': np.std(results['fd_sdr_times']),
        'error_mean': np.mean(results['fd_sdr_errors']),
        'error_std': np.std(results['fd_sdr_errors'])
    }
    
    # Print performance table
    print("\nPERFORMANCE METRICS:")
    print("-"*80)
    print(f"{'Method':<10}{'Time (mean ± std)':<30}{'Error (mean ± std)':<30}")
    print("-"*80)
    
    for method in methods:
        m = stats[method]
        time_str = f"{m['time_mean']:.4f}s ± {m['time_std']:.4f}"
        error_str = f"{m['error_mean']:.4f} ± {m['error_std']:.4f}"
        print(f"{method:<10}{time_str:<30}{error_str:<30}")
    
    # Footer
    print("="*80 + "\n")


In [110]:
import pickle
import os
from datetime import datetime
from glob import glob

def save_results(results, config=None, result_name="simulation_results", directory='results'):
    """
    Save simulation results to [result_name].pkl file
    
    Parameters:
    -----------
    results : dict
        Dictionary containing simulation results
    config : dict, optional
        Dictionary containing simulation configuration  
    result_name : str
        Base name for the results file (without extension)
        Default: "simulation_results"
    directory : str, optional
        Directory to save results (default: 'results')
    
    Returns:
    --------
    str
        Full path to the saved .pkl file
        
    Example:
    --------
    >>> save_results(results, config, "A")
    'results/A.pkl'
    """
    # Create directory if it doesn't exist
    os.makedirs(directory, exist_ok=True)
    
    # Create filename with .pkl extension
    filename = f"{result_name}.pkl"
    filepath = os.path.join(directory, filename)
    
    # Prepare data to save
    data = {
        'results': results,
        'config': config,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    # Save to pickle file
    with open(filepath, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"Results saved to: {filepath}")
    # return filepath

def load_single_result(result_name="simulation_results", directory='results'):
    """
    Load simulation results from [result_name].pkl file
    
    Parameters:
    -----------
    result_name : str
        Base name for the results file (without extension)
        Default: "simulation_results"
    directory : str, optional
        Directory where results are saved (default: 'results')
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'results': the simulation results
        - 'config': the simulation configuration (if available)
        - 'timestamp': when the results were saved
        
    Raises:
    -------
    FileNotFoundError
        If the specified results file doesn't exist
        
    Example:
    --------
    >>> data = load_results("A")
    >>> results = data['results']
    >>> config = data['config']
    """
    # Create filename with .pkl extension
    filename = f"{result_name}.pkl"
    filepath = os.path.join(directory, filename)
    
    # Check if file exists
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No results file found at: {filepath}")
    
    # Load data from pickle file
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    # print(f"Results loaded from: {filepath}")
    return data

def load_all_results(directory='results'):
    """
    Load all results from the specified directory
    
    Parameters:
    -----------
    directory : str, optional
        Directory where results are saved (default: 'results')
    
    Returns:
    --------
    list
        List of dictionaries containing results and configurations
    """
    filepaths = sorted(glob(os.path.join(directory, '*.pkl')))
    results_list = []
    configs_list = []

    for filepath in filepaths:
        # Get filename without extension
        filename = os.path.splitext(os.path.basename(filepath))[0]
        data = load_single_result(filename, directory)
        results_list.append(data['results'])
        configs_list.append(data['config'])

    return results_list, configs_list

In [123]:
from tabulate import tabulate
import numpy as np

def results_to_table(results_list, configs_list, IF_TIME=False):
    """
    Convert multiple simulation results into a single formatted table.
    
    Parameters:
    -----------
    results_list : list of dict
        List of result dictionaries, each in format:
        {
            'gwire_errors': [0.1, 0.2, 0.3],    
            'fopg_errors': [0.2, 0.3, 0.4],
            'fd_sdr_errors': [0.3, 0.4, 0.5],
            'gwire_times': [0.1, 0.2, 0.3],
            'fopg_times': [0.2, 0.3, 0.4],
            'fd_sdr_times': [0.3, 0.4, 0.5]
        }
    configs_list : list of dict
        List of configuration dictionaries corresponding to each result
    
    Returns:        
    --------
    str
        Formatted table as string
    """
    # Helper function to handle empty lists
    def format_error(errors):
        if not errors:  # if list is empty
            return "N/A"
        return f"{np.mean(errors):.2f}({np.std(errors):.2f})"

    # Helper function to get model description
    def get_model_description(config):
        mode_X = config['mode_X']
        mode_y = config['mode_y']
        model = mode_y[:2] + ' - ' + mode_X[1:]
        # if mode_y in ['(3)', '(4)']:
        #     model += f', α={config["alpha"]}'
        return model

    # Table headers
    headers = ["(n,p)", "Model", "Fd-SDR", "FOPG", "GWIRE"]
    
    table_data = []
    for result, config in zip(results_list, configs_list):
        n = config['n']
        p = config['p']
        model = get_model_description(config)
        
        if IF_TIME:
            row = [
                f"({n},{p})",
                model,
                format_error(result.get('fd_sdr_times', [])),
                format_error(result.get('fopg_times', [])),
                format_error(result.get('gwire_times', []))
            ]
        else:
            row = [
                f"({n},{p})",
                model,
                format_error(result.get('fd_sdr_errors', [])),
                format_error(result.get('fopg_errors', [])),
                format_error(result.get('gwire_errors', []))
            ]
        table_data.append(row)
    
    print(tabulate(table_data, headers=headers, tablefmt="grid", stralign="center"))
    # return table_data

# Example usage:
# results_to_table([results_200_10], [config])

In [116]:
# Single simulation
np.random.seed(123)

config = {
    'n': 200,
    'p': 10,
    'mode_X': '(a)',
    'mode_y': '(1)',
    'num_repeats': 5,
    'metric': 'Geodesic',
    'IF_GWIRE': False,
    'verbose': True
}

results = run_simulation(config = config)
result_name = f"results-{config['n']}-{config['p']}-{config['mode_X']}-{config['mode_y']}"
save_results(results, config, result_name=result_name, directory='results-III')


                    Starting Simulation                     
Configuration:
- Repeats: 5
- Dimensions: n=200, p=10
- X mode: (a), Y mode: (1)
- Metric: Geodesic


Iteration 1/5 ---------------
FOPG:  Time=1.767s, Error=0.0289
FD-SDR: Time=0.549s, Error=0.0429
----------------------------------------
Iteration completed in 2.32 seconds
----------------------------------------

Iteration 2/5 ---------------
FOPG:  Time=1.359s, Error=0.0452
FD-SDR: Time=0.545s, Error=0.0832
----------------------------------------
Iteration completed in 1.91 seconds
----------------------------------------

Iteration 3/5 ---------------
FOPG:  Time=1.423s, Error=0.0604
FD-SDR: Time=0.539s, Error=0.1149
----------------------------------------
Iteration completed in 1.97 seconds
----------------------------------------

Iteration 4/5 ---------------
FOPG:  Time=1.353s, Error=0.0876
FD-SDR: Time=0.541s, Error=0.1072
----------------------------------------
Iteration completed in 1.89 seconds
--------------

In [ ]:
# Define the parameter combinations to test
n_p_combinations = [(200, 10), (400, 20)]
# n_p_combinations = [(600, 100)]

# mode_combinations = [
#     ('(3)', '(a)'), ('(3)', '(b)')
# ]

mode_combinations = [
    ('(1)', '(a)'), ('(1)', '(b)'),
    ('(2)', '(a)'), ('(2)', '(b)'),
('(3)', '(a)'), ('(3)', '(b)')
]

# Base configuration
base_config = {
    'num_repeats': 100,
    'neigh': None,
    'metric': 'Wasserstein',
    'verbose': False
}

# Run simulations for all combinations
for n, p in n_p_combinations:
    if p > 20:
        IF_GWIRE = False
    else:
        IF_GWIRE = True
        
    for mode_y, mode_X in mode_combinations:
        config = base_config.copy()
        config.update({
            'n': n,
            'p': p,
            'mode_X': mode_X,
            'mode_y': mode_y,
            'IF_GWIRE': IF_GWIRE
        })
        
        results = run_simulation(config=config)
        result_name = f"results-{n}-{p}-{mode_y}-{mode_X}"
        save_results(results, config, result_name=result_name, directory='results-III')
        print(f"Completed: n={n}, p={p}, modes=({mode_X},{mode_y})")

Parameters: n = 600, p = 100, mode_X = (a), mode_y = (3)


Running simulation: 100%|██████████| 100/100 [28:37<00:00, 17.17s/iter]


Results saved to: results-III/results-600-100-(3)-(a).pkl
Completed: n=600, p=100, modes=((a),(3))
Parameters: n = 600, p = 100, mode_X = (b), mode_y = (3)


Running simulation: 100%|██████████| 100/100 [34:34<00:00, 20.75s/iter]

Results saved to: results-III/results-600-100-(3)-(b).pkl
Completed: n=600, p=100, modes=((b),(3))


In [124]:
results_list, configs_list = load_all_results(directory='results-III')

# Results Table for Errors
results_to_table(results_list, configs_list, IF_TIME=False)

+-----------+---------+------------+------------+---------+
|   (n,p)   |  Model  |   Fd-SDR   |    FOPG    |  GWIRE  |
+===========+=========+============+============+=========+
| (600,100) | (1 - a) | 0.23(0.12) | 0.28(0.14) |   N/A   |
+-----------+---------+------------+------------+---------+
| (600,100) | (1 - b) | 0.24(0.14) | 0.27(0.12) |   N/A   |
+-----------+---------+------------+------------+---------+
| (600,100) | (2 - a) | 0.92(0.56) | 0.78(0.37) |   N/A   |
+-----------+---------+------------+------------+---------+
| (600,100) | (2 - b) | 0.98(0.60) | 0.71(0.31) |   N/A   |
+-----------+---------+------------+------------+---------+
| (600,100) | (3 - a) | 1.15(0.41) | 1.32(0.40) |   N/A   |
+-----------+---------+------------+------------+---------+
| (600,100) | (3 - b) | 1.13(0.36) | 1.33(0.36) |   N/A   |
+-----------+---------+------------+------------+---------+


In [22]:
# Results Table for Runtime
results_to_table(results_list, configs_list, IF_TIME=True)

+----------+---------+------------+------------+---------+
|  (n,p)   |  Model  |   Fd-SDR   |    FOPG    |  GWIRE  |
+==========+=========+============+============+=========+
| (200,10) | (1 - a) | 0.07(0.01) | 1.53(0.06) |   N/A   |
+----------+---------+------------+------------+---------+
| (200,10) | (1 - b) | 0.09(0.00) | 1.62(0.00) |   N/A   |
+----------+---------+------------+------------+---------+
